# Demo 1 — Streaming Schema Evolution

This notebook demonstrates additive schema evolution for the Event Hub stream.

It will:

1. send new crypto events with extra fields;
2. read only the new Event Hub offsets by reusing the existing checkpoint;
3. parse the expanded JSON schema;
4. evolve the existing Bronze Delta table with new columns.

The original event fields remain unchanged. The new fields are:

- `change_pct_24h`
- `high_price_24h`
- `low_price_24h`
- `volume_24h`

Older Bronze rows will contain `NULL` for these new columns.


## 1. Load shared configuration

In [0]:
%run ../config/00_config


## 2. Install and import required libraries

Run the installation cell when `azure-eventhub` is not already available.


In [0]:
%pip install azure-eventhub


In [0]:
import json
import re
import time
import uuid
from datetime import datetime, timezone
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

from azure.eventhub import EventData, EventHubProducerClient
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    StringType,
    StructField,
    StructType,
)


## 3. Retrieve the Event Hub connection string

In [0]:
try:
    eventhub_connection_string = dbutils.secrets.get(
        scope=eventhub_secret_scope,
        key=eventhub_secret_key,
    )
except Exception as exc:
    raise RuntimeError(
        "Could not retrieve the Event Hub connection string. "
        f"Check scope '{eventhub_secret_scope}' and "
        f"key '{eventhub_secret_key}'."
    ) from exc

if not eventhub_connection_string:
    raise RuntimeError("The Event Hub connection string is empty.")

print("Event Hub connection string retrieved successfully.")


## 4. Define the Binance 24-hour ticker API

This endpoint returns the latest price together with 24-hour market statistics.


In [0]:
binance_24h_url = (
    "https://data-api.binance.vision/api/v3/ticker/24hr"
)

producer_id = "demo1_binance_evolved_producer"
number_of_cycles = 5
polling_interval_seconds = 5
request_timeout_seconds = 15

print(f"Symbols: {historical_symbols}")
print(f"Cycles: {number_of_cycles}")
print(
    f"Expected evolved events: "
    f"{number_of_cycles * len(historical_symbols)}"
)


## 5. Create a function that returns the expanded market data

In [0]:
def get_24h_market_data(symbol: str) -> dict:
    request_url = (
        f"{binance_24h_url}?"
        f"{urlencode({'symbol': symbol})}"
    )

    try:
        with urlopen(
            request_url,
            timeout=request_timeout_seconds,
        ) as response:
            payload = json.loads(
                response.read().decode("utf-8")
            )
    except HTTPError as exc:
        raise RuntimeError(
            f"Binance returned HTTP {exc.code} for {symbol}."
        ) from exc
    except URLError as exc:
        raise RuntimeError(
            f"Could not reach Binance for {symbol}: "
            f"{exc.reason}"
        ) from exc

    required_fields = [
        "lastPrice",
        "priceChangePercent",
        "highPrice",
        "lowPrice",
        "volume",
    ]

    missing_fields = [
        field
        for field in required_fields
        if field not in payload
    ]

    if missing_fields:
        raise ValueError(
            f"Binance response for {symbol} is missing: "
            f"{missing_fields}"
        )

    return {
        "price_usd": float(payload["lastPrice"]),
        "change_pct_24h": float(
            payload["priceChangePercent"]
        ),
        "high_price_24h": float(payload["highPrice"]),
        "low_price_24h": float(payload["lowPrice"]),
        "volume_24h": float(payload["volume"]),
    }


## 6. Create the evolved event structure

The first six fields match the original schema. Four new fields are added.


In [0]:
def create_evolved_event(
    symbol: str,
    market_data: dict,
) -> dict:
    return {
        "event_id": str(uuid.uuid4()),
        "symbol": symbol,
        "price_usd": market_data["price_usd"],
        "event_time": datetime.now(
            timezone.utc
        ).isoformat(),
        "producer_id": producer_id,
        "source_system": source_system_streaming,
        "change_pct_24h": market_data[
            "change_pct_24h"
        ],
        "high_price_24h": market_data[
            "high_price_24h"
        ],
        "low_price_24h": market_data[
            "low_price_24h"
        ],
        "volume_24h": market_data["volume_24h"],
    }

sample_market_data = get_24h_market_data("BTCUSDT")
sample_evolved_event = create_evolved_event(
    "BTCUSDT",
    sample_market_data,
)

print(json.dumps(sample_evolved_event, indent=2))


## 7. Send evolved events to Event Hubs

Five cycles with three symbols create 15 new events.


In [0]:
producer_client = (
    EventHubProducerClient.from_connection_string(
        conn_str=eventhub_connection_string,
        eventhub_name=eventhub_name,
    )
)

sent_evolved_events = []

try:
    with producer_client:
        for cycle_number in range(
            1,
            number_of_cycles + 1,
        ):
            event_batch = producer_client.create_batch()
            cycle_events = []

            for symbol in historical_symbols:
                market_data = get_24h_market_data(
                    symbol
                )
                event = create_evolved_event(
                    symbol,
                    market_data,
                )

                event_batch.add(
                    EventData(json.dumps(event))
                )
                cycle_events.append(event)

            producer_client.send_batch(event_batch)
            sent_evolved_events.extend(cycle_events)

            print(
                f"Cycle {cycle_number}/"
                f"{number_of_cycles}: "
                f"sent {len(cycle_events)} events"
            )

            if cycle_number < number_of_cycles:
                time.sleep(
                    polling_interval_seconds
                )

except Exception as exc:
    raise RuntimeError(
        "Failed to send evolved Event Hub events."
    ) from exc

print(
    f"Total evolved events sent: "
    f"{len(sent_evolved_events)}"
)


## 8. Define the expanded JSON schema

In [0]:
evolved_event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("symbol", StringType(), False),
    StructField("price_usd", DoubleType(), False),
    StructField("event_time", StringType(), False),
    StructField("producer_id", StringType(), False),
    StructField("source_system", StringType(), False),
    StructField(
        "change_pct_24h",
        DoubleType(),
        True,
    ),
    StructField(
        "high_price_24h",
        DoubleType(),
        True,
    ),
    StructField(
        "low_price_24h",
        DoubleType(),
        True,
    ),
    StructField(
        "volume_24h",
        DoubleType(),
        True,
    ),
])


## 9. Build the Event Hubs Kafka settings

The shaded Kafka login class is used for Shared/Standard Databricks compute.


In [0]:
endpoint_match = re.search(
    r"Endpoint=sb://([^/;]+)",
    eventhub_connection_string,
    flags=re.IGNORECASE,
)

if not endpoint_match:
    raise ValueError(
        "Could not extract Event Hub namespace "
        "from the connection string."
    )

eventhub_namespace_host = endpoint_match.group(1)
kafka_bootstrap_servers = (
    f"{eventhub_namespace_host}:9093"
)

escaped_connection_string = (
    eventhub_connection_string
    .replace("\\", "\\\\")
    .replace('"', '\\"')
)

kafka_sasl_jaas_config = (
    "kafkashaded.org.apache.kafka.common."
    "security.plain.PlainLoginModule required "
    'username="$ConnectionString" '
    f'password="{escaped_connection_string}";'
)

print(f"Kafka server: {kafka_bootstrap_servers}")
print(f"Event Hub topic: {eventhub_name}")
print(f"Checkpoint: {crypto_ticks_checkpoint_path}")


## 10. Read the new Event Hub events

The existing checkpoint is reused, so Spark resumes after the offsets already processed by `05_eventhub_consumer`.


In [0]:
raw_evolved_stream_df = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        kafka_bootstrap_servers,
    )
    .option("subscribe", eventhub_name)
    .option(
        "kafka.security.protocol",
        "SASL_SSL",
    )
    .option(
        "kafka.sasl.mechanism",
        "PLAIN",
    )
    .option(
        "kafka.sasl.jaas.config",
        kafka_sasl_jaas_config,
    )
    .option(
        "kafka.request.timeout.ms",
        "60000",
    )
    .option(
        "kafka.session.timeout.ms",
        "30000",
    )
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

print("Evolved Event Hub stream created.")


## 11. Parse the expanded JSON events

In [0]:
evolved_stream_df = (
    raw_evolved_stream_df
    .withColumn(
        "raw_json",
        F.col("value").cast("string"),
    )
    .withColumn(
        "parsed_event",
        F.from_json(
            F.col("raw_json"),
            evolved_event_schema,
        ),
    )
    .select(
        F.col(
            "parsed_event.event_id"
        ).alias("event_id"),
        F.col(
            "parsed_event.symbol"
        ).alias("symbol"),
        F.col(
            "parsed_event.price_usd"
        ).alias("price_usd"),
        F.to_timestamp(
            F.col("parsed_event.event_time")
        ).alias("event_time"),
        F.col(
            "parsed_event.producer_id"
        ).alias("producer_id"),
        F.col(
            "parsed_event.source_system"
        ).alias("source_system"),
        F.col(
            "parsed_event.change_pct_24h"
        ).alias("change_pct_24h"),
        F.col(
            "parsed_event.high_price_24h"
        ).alias("high_price_24h"),
        F.col(
            "parsed_event.low_price_24h"
        ).alias("low_price_24h"),
        F.col(
            "parsed_event.volume_24h"
        ).alias("volume_24h"),
        F.col("partition").alias(
            "eventhub_partition"
        ),
        F.col("offset").alias(
            "eventhub_offset"
        ),
        F.col("timestamp").alias(
            "eventhub_enqueued_at"
        ),
        F.col("raw_json"),
        F.current_timestamp().alias(
            "ingested_at"
        ),
    )
)

valid_evolved_stream_df = (
    evolved_stream_df.filter(
        F.col("event_id").isNotNull()
        & F.col("symbol").isNotNull()
        & F.col("price_usd").isNotNull()
        & F.col("event_time").isNotNull()
    )
)


## 12. Evolve and append to the Bronze table

`mergeSchema = true` adds the four new columns to the existing Delta table.

The same checkpoint prevents the original 30 events from being processed again.


In [0]:
spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "true",
)

evolution_query = (
    valid_evolved_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        crypto_ticks_checkpoint_path,
    )
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(streaming_bronze_table)
)

evolution_query.awaitTermination()

print("Schema evolution ingestion completed.")


## 13. Validate the evolved Bronze table

Expected behavior:

- original rows remain;
- new rows are appended;
- the new columns appear in Unity Catalog;
- original rows contain `NULL` in the new columns;
- evolved rows contain 24-hour market statistics.


In [0]:
evolved_bronze_df = spark.table(
    streaming_bronze_table
)

display(
    evolved_bronze_df
    .select(
        "event_id",
        "symbol",
        "price_usd",
        "event_time",
        "change_pct_24h",
        "high_price_24h",
        "low_price_24h",
        "volume_24h",
        "producer_id",
    )
    .orderBy(
        F.col("event_time").desc()
    )
)

schema_summary_df = (
    evolved_bronze_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias("total_events"),
        F.count(
            "change_pct_24h"
        ).alias("evolved_events"),
        F.count(
            F.when(
                F.col("change_pct_24h").isNull(),
                1,
            )
        ).alias("original_events"),
    )
    .orderBy("symbol")
)

display(schema_summary_df)

print("Current Bronze schema:")
evolved_bronze_df.printSchema()


## Board explanation

> The first producer version sent six fields. The evolved producer added four optional market-statistic fields. The consumer reused the same checkpoint, processed only new Event Hub offsets, and used Delta schema merging to add the new columns. Existing records were preserved and contain null values for fields that did not exist when they were produced.

## Next notebook

`validation/07_validation.ipynb`
